# Part 1 – SAR Image Despeckling Using DiffusionSAR

This notebook compares a classical **boxcar filter** with the deep-learning model **DiffusionSAR**
(Ma et al., 2024) on a NISAR HH amplitude image.

**How to run (Google Colab, nothing to install):**
1. *Runtime → Change runtime type →* select **T4 GPU** → *Save*. (Without a GPU it still works, but takes ~10–15 min.)
2. *Runtime → Run all*.

The image, the pre-trained model and the code are downloaded automatically (~170 MB, a few seconds).
The three images are shown at the bottom of the notebook (about 1 minute on the GPU).

In [ ]:
# 1. Get the data, model and code (Part1_Denoising.zip)
import os, sys, glob, zipfile, urllib.request

ZIP_URL = "https://github.com/dearest-D/SAR-DL-Practical/releases/download/v1.0/Part1_Denoising.zip"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    os.chdir("/content")

if not os.path.exists("DiffusionSAR.py"):
    if not os.path.isdir("Part1_Denoising"):
        zips = sorted(glob.glob("Part1_Denoising*.zip"))
        if not zips:
            try:
                print("Downloading", ZIP_URL)
                urllib.request.urlretrieve(ZIP_URL, "Part1_Denoising.zip")
                zips = ["Part1_Denoising.zip"]
            except Exception as err:
                print("Download failed:", err)
        if not zips and IN_COLAB:
            from google.colab import files
            print("Please select Part1_Denoising.zip ...")
            zips = list(files.upload().keys())
        if not zips:
            raise FileNotFoundError("Part1_Denoising.zip not found - put it next to this notebook.")
        zipfile.ZipFile(zips[0]).extractall(".")
    os.chdir("Part1_Denoising")
print("Working folder:", os.getcwd())
print(sorted(os.listdir(".")))

In [ ]:
# 2. Install the one missing package and check for a GPU
%pip install -q tifffile
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "-> DiffusionSAR takes about 1 minute")
else:
    print("No GPU -> running on CPU (about 10-15 minutes).")
    print("For a GPU in Colab: Runtime > Change runtime type > T4 GPU, then Run all again.")

In [ ]:
# 3. Despeckle with the boxcar filter and DiffusionSAR, and show the results side by side
%matplotlib inline
%run DiffusionSAR.py

### Try it yourself
Change `window` in the cell below and re-run only that cell (takes < 1 s).
Results (GeoTIFF + PNG) are saved in the `results` folder (*Files* panel on the left).

In [ ]:
# Only the boxcar filter is recomputed here; the DiffusionSAR result is read from results/
import numpy as np, tifffile, matplotlib.pyplot as plt
import DiffusionSAR as D

window = 9                                  # <- change me (3, 5, 7, 9, 11, ...)

amp = tifffile.imread("NISAR_HH_amplitude.tif")
box = D.boxcar(amp, window)
dif = tifffile.imread("results/DiffusionSAR.tif")

fig, axes = plt.subplots(1, 3, figsize=(16, 5.8))
for ax, img, title in zip(axes, [amp, box, dif],
                          ["Original (noisy)", f"Boxcar {window}x{window}", "DiffusionSAR"]):
    vmin, vmax = np.percentile(img, [2, 98])
    ax.imshow(img, cmap="gray", vmin=vmin, vmax=vmax); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()